# Movie Categorization with PCA and K-Means

This notebook categorizes movies based on their features using unsupervised learning techniques.

## Steps
1. **Load Data**: Import movie features, taxonomy, and genre data.
2. **Preprocess**: Pivot the data to a wide format (Movies x Features) and normalize it.
3. **PCA**: Reduce dimensionality to visualize and compress the feature space.
4. **Clustering**: Apply K-Means to group similar movies.
5. **Analysis**: Visualize and interpret the resulting clusters.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
sns.set(style='whitegrid')

In [ ]:
# file paths
BASE_DIR = '/Users/lewiswilliamcampbell/Desktop/Project 1 ANTI'
FEATURE_FILE = os.path.join(BASE_DIR, 'feature_data_longform.csv')
TAXONOMY_FILE = os.path.join(BASE_DIR, 'feature_taxonomy.csv')
GENRE_FILE = os.path.join(BASE_DIR, 'genre_data.csv')
OUTPUT_CLUSTERS_FILE = os.path.join(BASE_DIR, 'movie_clusters.csv')

## 1. Load Data

In [ ]:
print("Loading data...")
features_df = pd.read_csv(FEATURE_FILE)
taxonomy_df = pd.read_csv(TAXONOMY_FILE)
genre_df = pd.read_csv(GENRE_FILE)

# Check for duplicates in genre_data
if genre_df['movie_id'].duplicated().any():
    print("Warning: Duplicate movie_ids found in genre_data. Dropping duplicates.")
    genre_df = genre_df.drop_duplicates(subset=['movie_id'])

print(f"Features shape: {features_df.shape}")
print(f"Taxonomy shape: {taxonomy_df.shape}")
print(f"Genre shape: {genre_df.shape}")

## 2. Data Preprocessing & Pivoting

In [ ]:
print("Pivoting data to (Movies x Features)...")
# Pivot: Index=imdb_id, Columns=feature_id, Values=trigger
pivoted_df = features_df.pivot_table(index='imdb_id', columns='feature_id', values='trigger', fill_value=0, aggfunc='max')

# Map feature_ids to names
feature_map = dict(zip(taxonomy_df['feature_id'], taxonomy_df['feature']))
pivoted_df.columns = [feature_map.get(col, f'feature_{col}') for col in pivoted_df.columns]

print(f"Pivoted Data Shape: {pivoted_df.shape}")
pivoted_df.head()

In [ ]:
# Normalize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(pivoted_df)

## 3. Principal Component Analysis (PCA)

In [ ]:
# Run PCA
pca = PCA(n_components=2) # Start with 2 for visualization
pca_data = pca.fit_transform(scaled_data)

explained_variance = pca.explained_variance_ratio_
print(f"Explained Variance Ratio (PC1, PC2): {explained_variance}")
print(f"Total Explained Variance: {sum(explained_variance):.2f}")

## 4. K-Means Clustering

In [ ]:
# Determine optimal clusters (Optional: Elbow Method could be added here)
# For now, let's start with K=5 based on common genres
n_clusters = 5

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(pca_data)

## 5. Visualization & Analysis

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=clusters, palette='viridis', s=50, alpha=0.7)
plt.title(f'Movie Clusters (K={n_clusters}) on PCA Components')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# Save Results
results_df = pd.DataFrame(index=pivoted_df.index)
results_df['cluster'] = clusters

# Merge with genre info
results_df = results_df.merge(genre_df[['movie_id', 'movie_name', 'genre']], left_index=True, right_on='movie_id', how='left')
cols = ['movie_id', 'movie_name', 'cluster', 'genre']
results_df = results_df[cols]

results_df.to_csv(OUTPUT_CLUSTERS_FILE, index=False)
print(f"Saved clustered data to {OUTPUT_CLUSTERS_FILE}")

# Preview clusters
results_df.head()

In [ ]:
# Inspect a specific cluster (e.g., Cluster 0)
cluster_id = 0
print(f"Movies in Cluster {cluster_id}:")
print(results_df[results_df['cluster'] == cluster_id][['movie_name', 'genre']].head(10))